# 🔭 SPECTRA — Auto-Resuming GPU Training

**Bharatiya Antariksh Hackathon 2026 | PS 10**

**INSTRUCTIONS:**
1. Zip your `Spectra-ISRO` folder (make sure to delete the `raw` folder first!)
2. Upload `Spectra-ISRO.zip` to your Google Drive.
3. Go to **Runtime → Change runtime type → select T4 GPU**.
4. Go to **Runtime → Run all**.

In [ ]:
# ─── 1. Check GPU ──────────────────────────
import torch
if torch.cuda.is_available():
    print(f'✅ GPU Active: {torch.cuda.get_device_name(0)}')
else:
    print('⚠️ No GPU detected! Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# ─── 2. Setup Google Drive & Safe Checkpoints 
from google.colab import drive
import os

drive.mount('/content/drive')

# Unzip from Google Drive directly into Colab's ultra-fast SSD
!rm -rf /content/Spectra-ISRO
!unzip -q /content/drive/MyDrive/Spectra-ISRO.zip -d /content/

%cd "/content/Spectra-ISRO"
!pip install -r ml/requirements.txt

# VERY IMPORTANT FIX: 
# We symlink the checkpoints folder DIRECTLY to your Google Drive.
# If Colab disconnects at Epoch 109, you will NOT lose your data!
!mkdir -p /content/drive/MyDrive/Spectra-Trained-Models
!rm -rf ml/checkpoints
!ln -s /content/drive/MyDrive/Spectra-Trained-Models ml/checkpoints
print("✅ Unzipped and Checkpoints are safely linked to Google Drive!")

In [ ]:
# ─── 3. Train Model A: Super-Resolution ────
import os
import glob

def run_training(mode):
    # Check if best model already exists (meaning training is 100% done)
    if os.path.exists(f'ml/checkpoints/{mode}_best.pth'):
        print(f"✅ {mode.upper()} model is already fully trained! Skipping.")
        return
        
    # Find the latest checkpoint to resume from
    checkpoints = sorted(glob.glob(f'ml/checkpoints/{mode}_epoch_*.pth'))
    cmd = f"python ml/train.py --mode {mode}"
    
    if checkpoints:
        latest = checkpoints[-1]
        print(f"🔄 Found checkpoint! Resuming from: {latest}")
        cmd += f" --resume {latest}"
    else:
        print(f"🚀 Starting {mode.upper()} training from scratch...")
        
    os.system(cmd)

run_training('sr')

In [ ]:
# ─── 4. Train Model B: Colorization ────────
run_training('colorize')

In [ ]:
# ─── 5. Train End-to-End SpectraSatNet ─────
run_training('e2e')

In [ ]:
# ─── 6. Export ONNX ────────────────────────
if os.path.exists('ml/checkpoints/e2e_best.pth'):
    !python ml/export_onnx.py --model e2e --checkpoint ml/checkpoints/e2e_best.pth --output ml/checkpoints/spectrasatnet.onnx
    print("✅ Final ONNX model saved directly to Google Drive!")
else:
    print("⏳ Waiting for end-to-end model to finish before exporting ONNX.")